# Tier-C (part 2) — Hyperparameter Optimization + Class-imbalance study

Two studies, both **dual-GPU** and **restart-safe from S3**:

**A. Optuna HPO** — tunes the headline models (ConvMambaLOB, MambaLOB, TLOB) on NIFTY $k{=}100$. Dual-GPU by
running one *model's* study per GPU (two models search concurrently; no SQLite write contention). Resumable
for free: each study uses an on-disk SQLite DB that we pull from / push to S3, and Optuna's `load_if_exists`
skips trials already in the DB. A killed session just re-pulls the DB and continues.

**B. Class-imbalance study** — on the best model (ConvMambaLOB), compares loss/sampling strategies
(`weighted_ce`, `plain_ce`, `focal`, `label_smoothing`, `balanced_sampling`) by their effect on macro-$F_1$
and **per-class recall** of the minority directional classes. Reuses the dual-GPU sharded runner with the
*fixed* per-run S3 resume (pulls merged **and** shard CSVs — the bug that re-ran everything in the Tier-B run
is fixed here; see the note in cell 8).

Setup: **GPU T4 x2**; secrets `GH_PAT`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`.
S3 layout: `experiments/hpo/` and `experiments/imbalance/`.


## 1. Runtime check (expect 2 GPUs)

In [ ]:
import torch, platform
print("Python:", platform.python_version(), "| Torch:", torch.__version__)
n = torch.cuda.device_count()
print(f"GPUs visible: {n}")
for i in range(n):
    print(f"  cuda:{i} = {torch.cuda.get_device_name(i)}")
assert n >= 1, "Enable GPU. For 2x throughput choose 'GPU T4 x2'."

## 2. Get the project code (GH_PAT secret)

In [ ]:
import sys, subprocess, pathlib
REPO_URL = "https://github.com/rajjoseph48/nse-lob-capstone.git"
REPO_DIR = "nse-lob-capstone"
def _get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v: return v
    except Exception: pass
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception: pass
    import os
    return os.environ.get(name, "")
GITHUB_TOKEN = _get_secret("GH_PAT")
def add_modeling(base):
    base = pathlib.Path(base)
    for cand in (base / "modeling", base):
        if (cand / "models.py").exists():
            sys.path.insert(0, str(cand.resolve())); return str(cand.resolve())
    return None
MODELING_DIR = None
for c in (".", REPO_DIR, "/kaggle/working/" + REPO_DIR, "/content/" + REPO_DIR):
    MODELING_DIR = add_modeling(c)
    if MODELING_DIR: break
if not MODELING_DIR:
    url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
    subprocess.run(["git", "clone", "--depth", "1", url], check=True)
    MODELING_DIR = add_modeling(REPO_DIR)
print("modeling/ dir:", MODELING_DIR)

## 3. Install deps (Mamba kernel + boto3 + optuna)

In [ ]:
import subprocess, sys
def pipq(*pkgs): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
pipq("boto3", "optuna", "scipy")
pipq("ninja", "packaging", "setuptools", "wheel")
pipq("--no-build-isolation", "causal-conv1d")
pipq("--no-build-isolation", "mamba-ssm")
import optuna
print("optuna", optuna.__version__)
try:
    import mamba_ssm; print("mamba-ssm", mamba_ssm.__version__)
except Exception as e:
    print("mamba-ssm NOT importable -> pure-PyTorch fallback:", repr(e))

## 4. Download NSE data from S3

In [ ]:
import os, boto3, pathlib
os.environ["AWS_ACCESS_KEY_ID"] = _get_secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = _get_secret("AWS_SECRET_ACCESS_KEY")
BUCKET, PREFIX, REGION = "lob-capstone-data", "lob-data/dhan/", "ap-south-2"
DATA_DIR_NSE = "nse_data/dhan"; pathlib.Path(DATA_DIR_NSE).mkdir(parents=True, exist_ok=True)
s3 = boto3.client("s3", region_name=REGION)
n = 0
for o in s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX).get("Contents", []):
    if o["Size"] == 0: continue
    dst = os.path.join(DATA_DIR_NSE, o["Key"].split("/")[-1])
    if not os.path.exists(dst): s3.download_file(BUCKET, o["Key"], dst)
    n += 1
print(f"{n} parquet files in {DATA_DIR_NSE}")

# Part A — Optuna hyperparameter search
## 5. HPO config + resume (pull existing study DBs from S3)

In [ ]:
import pathlib
from nbenv import s3_client, s3_pull_area

HPO_MODELS = ["convmambalob", "mambalob", "tlob"]   # headline models to tune
HPO_SYMBOL, HPO_H, HPO_FS = "NIFTY", 100, "all"
N_TRIALS = 20                                        # trials per model (TPE)
HPO_EPOCHS = 12                                      # short budget per trial
HPO_AREA = "hpo"
pathlib.Path("hpo").mkdir(exist_ok=True)

JOBS = [{"model": m, "study": f"{m}_{HPO_SYMBOL}_h{HPO_H}",
         "db": f"hpo/{m}_{HPO_SYMBOL}_h{HPO_H}.db"} for m in HPO_MODELS]

# resume: pull any existing DBs (Optuna load_if_exists then skips completed trials)
_s3 = s3_client()
for j in JOBS:
    got = s3_pull_area(_s3, pathlib.Path(j["db"]), HPO_AREA)
    print(f"{j['study']}: {'resumed from S3' if got else 'fresh study'}")
JOBS

## 6. Run HPO across both GPUs (one model per GPU) + periodic DB sync to S3
A small scheduler keeps both GPUs busy: it assigns queued model-studies to free GPUs, tails their logs, and
uploads each study DB to S3 every ~2 min and on completion — so a time-out costs at most the trials since the
last sync (and Optuna resumes the rest).

In [ ]:
import os, sys, subprocess, time, torch, pathlib
from nbenv import s3_client, s3_put_area

n_gpus = max(1, torch.cuda.device_count())
RUN_HPO = f"{MODELING_DIR}/run_hpo.py"

def _launch(gpu, job):
    env = dict(os.environ); env["CUDA_VISIBLE_DEVICES"] = str(gpu); env["PYTHONUNBUFFERED"] = "1"
    logf = f"hpo_{job['model']}.log"; fh = open(logf, "w")
    cmd = [sys.executable, "-u", RUN_HPO, "--model", job["model"], "--symbol", HPO_SYMBOL,
           "--horizon", str(HPO_H), "--feature-set", HPO_FS, "--storage", f"sqlite:///{job['db']}",
           "--study-name", job["study"], "--n-trials", str(N_TRIALS), "--epochs", str(HPO_EPOCHS),
           "--data-dir", DATA_DIR_NSE]
    print(f"launched HPO {job['study']} on GPU {gpu}")
    return {"proc": subprocess.Popen(cmd, env=env, stdout=fh, stderr=subprocess.STDOUT),
            "fh": fh, "logf": logf, "pos": 0, "job": job}

queue = list(JOBS); running = {}                      # gpu index -> live handle
last_sync = 0.0
while queue or running:
    # assign any queued study to a free GPU
    for g in range(n_gpus):
        if g not in running and queue:
            running[g] = _launch(g, queue.pop(0))
    # tail logs + reap finished
    for g, h in list(running.items()):
        with open(h["logf"]) as r:
            r.seek(h["pos"]); chunk = r.read(); h["pos"] = r.tell()
        if chunk: print(chunk, end="")
        if h["proc"].poll() is not None:              # finished -> flush, sync DB, free GPU
            with open(h["logf"]) as r: r.seek(h["pos"]); print(r.read(), end="")
            h["fh"].close(); s3_put_area(s3_client(), h["job"]["db"], HPO_AREA)
            print(f"   [{h['job']['study']}] done, DB synced to S3")
            del running[g]
    # periodic DB sync (crash-safety mid-run)
    if time.time() - last_sync > 120:
        for h in running.values():
            if pathlib.Path(h["job"]["db"]).exists():
                s3_put_area(s3_client(), h["job"]["db"], HPO_AREA)
        last_sync = time.time()
    time.sleep(10)
print("\nHPO complete.")

## 7. Best hyperparameters per model

In [ ]:
import optuna, json, pandas as pd
from nbenv import s3_client, s3_put_area
rows = []
for j in JOBS:
    st = optuna.load_study(study_name=j["study"], storage=f"sqlite:///{j['db']}")
    rows.append({"model": j["model"], "n_trials": len(st.trials),
                 "best_val_f1": round(st.best_value, 4), **st.best_params})
    print(f"{j['model']:14s} best val_f1={st.best_value:.4f}  {st.best_params}")
best = pd.DataFrame(rows)
best.to_json("hpo/best_params.json", orient="records", indent=2)
best.to_csv("hpo/best_params.csv", index=False)
s3_put_area(s3_client(), "hpo/best_params.json", HPO_AREA)
s3_put_area(s3_client(), "hpo/best_params.csv", HPO_AREA)
print("\nSaved best params to s3://.../experiments/hpo/results/")
best

# Part B — Class-imbalance study
## 8. Strategy grid + resume (FIXED per-run S3 resume)
The fix vs the Tier-B run: cell pulls the merged CSV **and** each per-shard CSV that `runner.run_one` pushes
after every run, so an interrupted session resumes at per-run granularity (the Tier-B notebook on Kaggle ran
a stale cell that pulled only the merged CSV — written only at the final merge — so it re-ran everything).
The resume key here includes `strategy`.

In [ ]:
import json, pathlib, pandas as pd
from nbenv import s3_client, s3_pull_area

AREA = "imbalance"
MERGED = pathlib.Path("results/imbalance.csv")
BEST_MODEL = "convmambalob"
STRATEGIES = ["weighted_ce", "plain_ce", "focal", "label_smoothing", "balanced_sampling"]
SYMBOL, HORIZONS = "NIFTY", [10, 100]               # h100 = worst imbalance; h10 for contrast

GRID = []
for strat in STRATEGIES:
    for h in HORIZONS:
        GRID.append({"model": BEST_MODEL, "symbol": SYMBOL, "feature_set": "all",
                     "horizon": h, "label_scheme": "A", "seed": 0, "strategy": strat})

_s3 = s3_client(); pathlib.Path("results").mkdir(exist_ok=True)
s3_pull_area(_s3, MERGED, AREA)
shard_csvs = [pathlib.Path(f"results/shard{i}.csv") for i in range(2)]
for sc in shard_csvs:
    s3_pull_area(_s3, sc, AREA)
done = set()
for f in [MERGED, *shard_csvs]:
    if f.exists():
        d = pd.read_csv(f)
        if "strategy" not in d.columns: d["strategy"] = "weighted_ce"
        done |= {(r.model, r.symbol, r.feature_set, int(r.horizon), r.label_scheme, int(r.seed), r.strategy)
                 for r in d.itertuples()}
def _key(s):
    return (s["model"], s["symbol"], s["feature_set"], int(s["horizon"]),
            s["label_scheme"], int(s["seed"]), s["strategy"])
todo = [s for s in GRID if _key(s) not in done]
pathlib.Path("runlist.json").write_text(json.dumps(todo))
print(f"grid={len(GRID)} | done={len(done)} | to run this session={len(todo)}")
todo

## 9. Launch shards (dual-GPU, reuses run_shard.py + per-run S3 push)

In [ ]:
import os, sys, subprocess, time, torch, pathlib
n_gpus = torch.cuda.device_count(); nshards = max(1, n_gpus)
RUN_SHARD = f"{MODELING_DIR}/run_shard.py"
pathlib.Path("results").mkdir(exist_ok=True)
procs = []
for i in range(nshards):
    env = dict(os.environ); env["CUDA_VISIBLE_DEVICES"] = str(i); env["PYTHONUNBUFFERED"] = "1"
    logf = f"shard{i}.log"; fh = open(logf, "w")
    cmd = [sys.executable, "-u", RUN_SHARD, "--runlist", "runlist.json", "--shard", str(i),
           "--nshards", str(nshards), "--data-dir", DATA_DIR_NSE, "--out", f"results/shard{i}.csv",
           "--ckpt-dir", "checkpoints/imb", "--area", "imbalance", "--epochs", "20"]
    procs.append([subprocess.Popen(cmd, env=env, stdout=fh, stderr=subprocess.STDOUT), fh, logf, 0])
    print(f"launched shard {i} on CUDA_VISIBLE_DEVICES={i}")
while any(p[0].poll() is None for p in procs):
    for p in procs:
        with open(p[2]) as r: r.seek(p[3]); chunk = r.read(); p[3] = r.tell()
        if chunk: print(chunk, end="")
    time.sleep(10)
for p in procs:
    p[1].close()
    with open(p[2]) as r: r.seek(p[3]); print(r.read(), end="")
print("\nexit codes:", [p[0].poll() for p in procs])

## 10. Merge shard CSVs + push to S3

In [ ]:
import glob, pandas as pd
from nbenv import s3_client, s3_put_area
keys = ["model", "symbol", "feature_set", "horizon", "label_scheme", "seed", "strategy"]
frames = [pd.read_csv(MERGED)] if MERGED.exists() else []
frames += [pd.read_csv(f) for f in glob.glob("results/shard*.csv")]
alldf = pd.concat(frames, ignore_index=True).drop_duplicates(subset=keys, keep="last") if frames else pd.DataFrame()
alldf.to_csv(MERGED, index=False)
s3_put_area(s3_client(), MERGED, AREA)
print(f"merged {len(alldf)} rows -> {MERGED}")
alldf.sort_values(keys)

## 11. Findings — which strategy best recovers the minority directional classes?
Imbalance hurts the Down/Up classes (macro-$F_1$ << weighted-$F_1$). The win condition is *higher macro-$F_1$
and higher Down/Up recall* without collapsing accuracy.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
res = pd.read_csv(MERGED)
res = res[res.model == BEST_MODEL]
ORDER = ["weighted_ce", "plain_ce", "focal", "label_smoothing", "balanced_sampling"]
for h in sorted(res.horizon.unique()):
    sub = res[res.horizon == h].set_index("strategy").reindex(ORDER)
    print("=" * 64, f"\n{BEST_MODEL}  {SYMBOL}  k={h}\n" + "=" * 64)
    cols = ["test_weighted_f1", "test_macro_f1", "recall_down", "recall_stat", "recall_up"]
    print(sub[cols].round(4).to_string())
    base = sub.loc["weighted_ce", "test_macro_f1"]
    print(f"\nΔ macro-F1 vs weighted_ce baseline:")
    print((sub["test_macro_f1"] - base).round(4).to_string())

# bar plot of macro-F1 by strategy (k=100)
sub = res[res.horizon == max(res.horizon)].set_index("strategy").reindex(ORDER)
ax = sub[["test_macro_f1", "recall_down", "recall_up"]].plot.bar(figsize=(8, 4))
ax.set(title=f"{BEST_MODEL} {SYMBOL} k={max(res.horizon)} — imbalance strategies",
       ylabel="score"); ax.grid(True, axis="y", alpha=0.3)
plt.xticks(rotation=20, ha="right"); plt.tight_layout()
plt.savefig("results/fig_imbalance.png", dpi=150, bbox_inches="tight")
from nbenv import s3_client, s3_put_area
s3_put_area(s3_client(), "results/fig_imbalance.png", AREA); plt.show()

## 12. Summary
- **HPO** (Part A): best hyperparameters per model are on S3 at `experiments/hpo/results/best_params.csv`.
  Re-train the headline configs with these for the final report numbers (or feed them into the Tier-C
  multi-seed notebook by overriding the model kwargs).
- **Imbalance** (Part B): the strategy table (`experiments/imbalance/results/imbalance.csv`) shows whether
  focal / sampling / label-smoothing beat the default class-weighted CE on macro-$F_1$ and minority-class
  recall. Use the winner in the final model and the cost-aware backtest.
- Both studies are dual-GPU and resume from S3 (HPO via the Optuna DB; imbalance via the fixed per-run CSV
  resume), so long runs survive Kaggle time-outs.
